# Démonstration — concevoir une ingestion fiable

**Public.** Étudiants de M2 qui connaissent Python et les bases de pandas.

**Prérequis.** Lancer le notebook depuis la racine du dépôt ou depuis `seance2/`. Les données sont déjà présentes dans `seance2/tp/data/raw`. Aucun accès réseau ni serveur PostgreSQL n'est nécessaire.

**Objectifs.** Traduire un contrat d'ingestion en code, conserver une zone brute rejouable, contrôler la forme des données, gérer un repère incrémental et produire les traces d'une exécution.


## Parcours

1. Identifier la source et figer le brut
2. Passer d'une lecture implicite à un contrat explicite
3. Aplatir une réponse JSON sans perdre son sens
4. Simuler une ingestion incrémentale avec un repère
5. Mesurer, contrôler et tracer l'exécution
6. Relier le notebook à un script de production


In [ ]:
from __future__ import annotations

import hashlib
import json
import sqlite3
import time
from statistics import median
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
import polars as pl
import requests
from sqlalchemy import create_engine, text

def trouver_racine() -> Path:
    for candidat in (Path.cwd(), *Path.cwd().parents):
        if (candidat / 'seance2' / 'tp' / 'data' / 'raw').exists():
            return candidat
    raise FileNotFoundError('Lancer ce notebook depuis le dépôt du cours.')

RACINE = trouver_racine()
DATA = RACINE / 'seance2' / 'tp' / 'data'
CSV = DATA / 'raw' / 'accidents_2024' / 'caract-2024.csv'
JSON_PARIS = DATA / 'raw' / 'api_geo' / '2026-10-25' / 'communes_75.json'
DOSSIER_DEMO = DATA / 'tmp' / 'demo_ingestion'
DOSSIER_DEMO.mkdir(parents=True, exist_ok=True)

pd.__version__, CSV.exists(), JSON_PARIS.exists()


## 1. La source fait partie du contrat

Avant de lire les données, l'ingestion enregistre ce qu'elle reçoit. La taille et l'empreinte cryptographique permettent ensuite de prouver quel fichier a été traité. Cette étape soutient la reprise sur incident et l'audit.


In [ ]:
def sha256(path: Path) -> str:
    empreinte = hashlib.sha256()
    with path.open('rb') as fichier:
        for bloc in iter(lambda: fichier.read(1024 * 1024), b''):
            empreinte.update(bloc)
    return empreinte.hexdigest()

source = {'nom': CSV.name, 'taille_octets': CSV.stat().st_size, 'sha256': sha256(CSV)}
source


## 2. Une lecture implicite constitue une hypothèse cachée

Le fichier BAAC utilise le point-virgule, la virgule décimale et des codes qui doivent rester du texte. L'appel par défaut échoue ou produit une structure inutilisable selon la version du moteur CSV.


In [ ]:
try:
    lecture_implicite = pd.read_csv(CSV)
    diagnostic_implicite = {
        'résultat': 'lecture terminée, mais schéma incorrect',
        'colonnes': lecture_implicite.columns.tolist(),
    }
except Exception as erreur:
    diagnostic_implicite = {
        'résultat': type(erreur).__name__,
        'message': str(erreur).splitlines()[0],
    }
diagnostic_implicite


### Le contrat explicite

Les identifiants et codes géographiques restent des chaînes. Les modalités courtes utilisent des entiers qui acceptent les valeurs manquantes.


In [ ]:
TYPES_CARACT = {
    'Num_Acc': 'string', 'dep': 'string', 'com': 'string',
    'lum': 'Int8', 'col': 'Int8',
}

def lire_caracteristiques(path: Path) -> pd.DataFrame:
    return pd.read_csv(
        path, sep=';', decimal=',', encoding='utf-8',
        na_values=['-1', ' -1'], dtype=TYPES_CARACT,
    )

caracteristiques = lire_caracteristiques(CSV)
caracteristiques[['Num_Acc', 'dep', 'com', 'lat', 'long']].head()


### Échouer tôt

Les contrôles d'import portent sur la forme reçue. Ils arrêtent l'exécution avant que des données incomplètes contaminent les traitements suivants.


In [ ]:
COLONNES_ATTENDUES = {'Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'dep', 'com', 'lat', 'long'}

def controler_import(df: pd.DataFrame, *, minimum_lignes: int = 40_000) -> dict[str, object]:
    manquantes = sorted(COLONNES_ATTENDUES - set(df.columns))
    if manquantes:
        raise ValueError(f'caract : colonnes manquantes {manquantes}')
    if len(df) < minimum_lignes:
        raise ValueError(f'caract : {len(df)} lignes, minimum attendu {minimum_lignes}')
    if not isinstance(df['dep'].dtype, pd.StringDtype):
        raise TypeError(f"caract : dep doit rester du texte, reçu {df['dep'].dtype}")
    return {'lignes': len(df), 'colonnes': len(df.columns), 'dep_dtype': str(df['dep'].dtype)}

controler_import(caracteristiques)


### Une sortie idempotente et efficace

L'écriture remplace le même fichier Parquet. Une relance avec la même source produit le même jeu de données, au lieu d'ajouter des doublons.


In [ ]:
sortie_parquet = DOSSIER_DEMO / 'caracteristiques.parquet'
colonnes_utiles = ['Num_Acc', 'an', 'dep', 'com', 'lat', 'long']
caracteristiques[colonnes_utiles].to_parquet(sortie_parquet, index=False)
pd.DataFrame({
    'format': ['CSV source', 'Parquet démonstration'],
    'taille_Mo': [CSV.stat().st_size / 2**20, sortie_parquet.stat().st_size / 2**20],
}).round(2)


## 3. JSON : aplatir sans perdre le schéma

Une réponse JSON contient des objets imbriqués et des listes. `json_normalize` produit des colonnes tabulaires, mais le contrat doit préciser quelles structures resteront des listes et quelles valeurs seront extraites.


In [ ]:
communes_brutes = json.loads(JSON_PARIS.read_text(encoding='utf-8'))
communes = pd.json_normalize(communes_brutes).rename(columns={
    'codeDepartement': 'code_departement',
    'centre.coordinates': 'coordonnees',
})
communes[['nom', 'code', 'code_departement', 'population', 'coordonnees', 'codesPostaux']].head(3)


### Appel réseau robuste, démontré sans dépendre du réseau

La fonction fixe un délai maximal et vérifie le statut HTTP. La cellule définit le comportement de production sans effectuer d'appel externe pendant le cours.


In [ ]:
def appeler_api_geo(departement: str, *, timeout_s: float = 10.0) -> list[dict]:
    reponse = requests.get(
        'https://geo.api.gouv.fr/communes',
        params={
            'codeDepartement': departement,
            'fields': 'nom,code,codeDepartement,population,centre,codesPostaux,type',
        },
        timeout=timeout_s,
    )
    reponse.raise_for_status()
    return reponse.json()

len(communes_brutes), communes_brutes[0]['nom']


## 4. Ingestion incrémentale : le repère est un état métier

La démonstration utilise SQLite en mémoire. Le même principe s'applique à PostgreSQL : la requête lit uniquement les lignes dont `updated_at` dépasse le dernier repère publié.


In [ ]:
connexion = sqlite3.connect(':memory:')
connexion.executescript("""
CREATE TABLE commandes (
    id INTEGER PRIMARY KEY, statut TEXT NOT NULL,
    montant REAL NOT NULL, updated_at TEXT NOT NULL
);
INSERT INTO commandes VALUES
    (1, 'payee', 42.50, '2026-10-28T08:00:00Z'),
    (2, 'expediee', 89.00, '2026-10-28T08:05:00Z'),
    (3, 'annulee', 15.00, '2026-10-28T08:12:00Z');
""")

def extraire_increment(conn: sqlite3.Connection, repere: str) -> pd.DataFrame:
    requete = """
        SELECT id, statut, montant, updated_at FROM commandes
        WHERE updated_at > ? ORDER BY updated_at, id
    """
    return pd.read_sql_query(requete, conn, params=(repere,))

repere = '1970-01-01T00:00:00Z'
premier_increment = extraire_increment(connexion, repere)
repere = premier_increment['updated_at'].max()
repere, premier_increment


### Relance, nouvelle donnée et mise à jour tardive

Une relance immédiate lit zéro ligne. Après deux changements, la destination devra faire un *upsert* sur la clé `id`, pas un simple ajout.


In [ ]:
assert extraire_increment(connexion, repere).empty
connexion.executemany("""
    INSERT INTO commandes (id, statut, montant, updated_at) VALUES (?, ?, ?, ?)
    ON CONFLICT(id) DO UPDATE SET
        statut = excluded.statut, montant = excluded.montant,
        updated_at = excluded.updated_at
""", [
    (4, 'payee', 120.0, '2026-10-28T08:20:00Z'),
    (3, 'remboursee', 15.0, '2026-10-28T08:21:00Z'),
])
deuxieme_increment = extraire_increment(connexion, repere)
deuxieme_increment


### Patron PostgreSQL pour la production

Cette cellule définit une connexion et une requête paramétrée sans les exécuter. Le filtrage reste dans la base et `chunksize` borne la mémoire du processus Python.


In [ ]:
def moteur_postgres(url: str):
    return create_engine(url, pool_pre_ping=True)

REQUETE_INCREMENTALE = text("""
    SELECT id, statut, montant, updated_at FROM commandes
    WHERE updated_at > :repere ORDER BY updated_at, id
""")

def lots_postgres(engine, repere: str, *, taille_lot: int = 5_000):
    return pd.read_sql(
        REQUETE_INCREMENTALE, engine,
        params={'repere': repere}, chunksize=taille_lot,
    )

str(REQUETE_INCREMENTALE).strip()


## 5. Performance : mesurer avant de choisir

La première comparaison montre l'effet du mode de calcul : une boucle Python face à une opération vectorisée pandas. La seconde compare pandas et Polars sur la même agrégation. Les temps seuls ne suffisent pas : chaque benchmark vérifie d'abord que les résultats sont équivalents.


In [ ]:
echantillon = caracteristiques[['an']].dropna().head(20_000).copy()

debut = time.perf_counter()
ages_boucle = [2024 - int(ligne['an']) for _, ligne in echantillon.iterrows()]
temps_boucle = time.perf_counter() - debut

debut = time.perf_counter()
ages_vectorises = 2024 - echantillon['an']
temps_vectorise = time.perf_counter() - debut

assert ages_boucle == ages_vectorises.astype(int).tolist()
pd.DataFrame({
    'méthode': ['pandas iterrows', 'pandas vectorisé'],
    'temps_ms': [temps_boucle * 1_000, temps_vectorise * 1_000],
}).round(2)


### pandas et Polars sur la même agrégation

Le test compte les accidents et calcule la latitude moyenne par département. Les deux DataFrame sont préparés avant le chronométrage : le résultat mesure l'agrégation, pas le coût de conversion entre bibliothèques.

Un benchmark sur ce volume illustre la méthode de comparaison. Il ne constitue pas une règle générale : le résultat dépend des données, de l'opération et de la machine.


In [ ]:
donnees_pandas = (
    caracteristiques[['dep', 'lat']]
    .dropna()
    .astype({'dep': 'string', 'lat': 'float64'})
)
donnees_polars = pl.from_pandas(donnees_pandas)

def agreger_pandas() -> pd.DataFrame:
    return (
        donnees_pandas.groupby('dep', as_index=False)
        .agg(accidents=('dep', 'size'), latitude_moyenne=('lat', 'mean'))
        .sort_values('dep')
        .reset_index(drop=True)
    )

def agreger_polars() -> pl.DataFrame:
    return (
        donnees_polars
        .group_by('dep')
        .agg(
            pl.len().alias('accidents'),
            pl.col('lat').mean().alias('latitude_moyenne'),
        )
        .sort('dep')
    )

resultat_pandas = agreger_pandas()
resultat_polars = agreger_polars().to_pandas()
pd.testing.assert_frame_equal(
    resultat_pandas,
    resultat_polars,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
)
resultat_pandas.head()


In [ ]:
def temps_median_ms(fonction, *, repetitions: int = 30) -> float:
    mesures = []
    for _ in range(repetitions):
        debut = time.perf_counter()
        fonction()
        mesures.append((time.perf_counter() - debut) * 1_000)
    return median(mesures)

temps_pandas = temps_median_ms(agreger_pandas)
temps_polars = temps_median_ms(agreger_polars)

comparaison_moteurs = pd.DataFrame({
    'moteur': ['pandas', 'Polars'],
    'temps_médian_ms': [temps_pandas, temps_polars],
    'lignes_en_entrée': [len(donnees_pandas), len(donnees_polars)],
})
comparaison_moteurs['rapport_à_pandas'] = comparaison_moteurs['temps_médian_ms'] / temps_pandas
comparaison_moteurs.round({'temps_médian_ms': 2, 'rapport_à_pandas': 2})


**Interprétation.** Le moteur le plus rapide sur cette cellule n'est pas automatiquement le meilleur choix pour tout le pipeline. La décision tient aussi compte de la mémoire, des formats d'entrée, de l'intégration avec les bibliothèques suivantes et du coût de maintenance. Dans ce cours, nous conservons pandas ; Polars devient une option à évaluer lorsqu'un profilage montre un goulot d'étranglement réel.


### Traçabilité de l'exécution

Le journal permet de répondre à quatre questions : quelle source, quand, combien de lignes et quel résultat publié ? En production, ces champs deviennent des logs structurés et des métriques.


In [ ]:
journal = {
    'pipeline': 'demo_caracteristiques_baac',
    'execution_utc': datetime.now(UTC).isoformat(timespec='seconds'),
    'source': str(CSV.relative_to(RACINE)),
    'source_sha256': source['sha256'],
    'lignes_lues': len(caracteristiques),
    'lignes_ecrites': len(pd.read_parquet(sortie_parquet)),
    'sortie': str(sortie_parquet.relative_to(RACINE)),
    'statut': 'succes',
}
print(json.dumps(journal, ensure_ascii=False, indent=2))


## 6. Du notebook au script

Le notebook sert à comprendre la source et à tester le contrat. Le passage en production conserve les fonctions pures, externalise les paramètres et ajoute orchestration, secrets, alertes et reprise.

**Pièges à éviter**

- avancer le repère avant d'avoir publié la sortie ;
- ajouter les incréments sans clé d'upsert ;
- confondre contrôle de forme et validation métier ;
- dépendre d'un état caché du notebook.


## Pause de raisonnement

Une commande créée à 08:10 arrive dans la source après que le repère a atteint 08:21. La requête actuelle la lira-t-elle ? Proposez une stratégie pour absorber ce retard sans dupliquer la destination.


In [ ]:
# Correction discutée après les propositions du groupe.
strategie_retard = {
    'problème': "un repère strict sur l'heure métier peut manquer un événement arrivé tard",
    'solution': "relire une fenêtre de recouvrement, puis faire un upsert idempotent sur la clé",
    'exemple': "repère effectif = dernier repère - 15 minutes",
}
strategie_retard


## Synthèse

Une ingestion fiable matérialise un contrat : source identifiable, schéma explicite, contrôles précoces, sortie rejouable, état incrémental sûr et preuves d'exécution. Les TP reprennent ces principes avec un CSV public, une API JSON et PostgreSQL.
